In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Fri Aug  8 14:47:14 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.65.06              Driver Version: 580.65.06      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...    On  |   00000000:01:00.0 Off |                  N/A |
| 30%   59C    P2            240W /  320W |   13633MiB /  16376MiB |     83%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip3 install --upgrade --quiet pip
!pip3 install --upgrade --quiet datasets[audio] transformers accelerate evaluate jiwer tensorboard gradio 
!pip3 install librosa
!pip3 install ipywidgets
!pip3 install deepspeed
!pip3 install mpi4py
!pip3 install soundfile
!pip3 install python-dotenv

  Using cached deepspeed-0.17.4.tar.gz (1.6 MB)
  Preparing metadata (setup.py) ... error
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [8 lines of output]
      Traceback (most recent call last):
        File "<string>", line 2, in <module>
        File "<pip-setuptools-caller>", line 35, in <module>
        File "/tmp/pip-install-7dx0jekw/deepspeed_5ea3bdde92b94baeb02cabbffec5c4e4/setup.py", line 110, in <module>
          cuda_major_ver, cuda_minor_ver = installed_cuda_version()
        File "/tmp/pip-install-7dx0jekw/deepspeed_5ea3bdde92b94baeb02cabbffec5c4e4/op_builder/builder.py", line 51, in installed_cuda_version
          raise MissingCUDAException("CUDA_HOME does not exist, unable to compile CUDA op(s)")
      op_builder.builder.MissingCUDAException: CUDA_HOME does not exist, unable to compile CUDA op(s)
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

In [4]:
from huggingface_hub import login
login(hf_token)

In [5]:
from datasets import load_dataset, DatasetDict, concatenate_datasets

data = load_dataset("brunopbb/ufcg-labmet-fala-texto-main-final")

train = data['train']
test = data['test']

In [6]:
train

Dataset({
    features: ['audio', 'transcription'],
    num_rows: 512
})

In [7]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-medium", language="Portuguese", task="transcribe")


In [8]:
def prepare_dataset(batch):
    # Para carregar e reamostrar dados de áudio de 48kHz para 16kHz
    audio = batch["audio"]

    # Para calcular as características de entrada log-Mel a partir de um array de áudio de entrada
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # Para codificar texto alvo em IDs de rótulos, você precisará de um tokenizador.
    # O tokenizador mapeia cada caractere único no seu texto para um ID numérico.
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    #batch["attention_mask"] = [1] * len(batch["input_features"])
    return batch

In [9]:
from datasets import set_audio_backend
set_audio_backend("soundfile")

ImportError: cannot import name 'set_audio_backend' from 'datasets' (/home/bruno-dev/anaconda3/envs/fine-tunning/lib/python3.10/site-packages/datasets/__init__.py)

In [10]:
data = data.map(prepare_dataset, remove_columns=data.column_names["train"], num_proc=2)

Map (num_proc=2):   0%|          | 0/512 [00:00<?, ? examples/s]

NotImplementedError: Could not run 'torchcodec_ns::create_from_tensor' with arguments from the 'CPU' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'torchcodec_ns::create_from_tensor' is only available for these backends: [Meta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradXLA, AutogradMPS, AutogradXPU, AutogradHPU, AutogradLazy, AutogradMTIA, AutogradMAIA, AutogradMeta, Tracer, AutocastCPU, AutocastMTIA, AutocastMAIA, AutocastXPU, AutocastMPS, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor, FuncTorchVmapMode, Batched, VmapMode, FuncTorchGradWrapper, PythonTLSSnapshot, FuncTorchDynamicLayerFrontMode, PreDispatch, PythonDisp

In [10]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")
model.generation_config.language = "portuguese"
model.generation_config.task = "transcribe"
model.config.use_cache = False
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None

#model.config.dropout = 0.1
#model.config.attention_dropout = 0.1


In [11]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Extrai e preenche as features de entrada.
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(
            input_features, padding="longest", return_tensors="pt"
        )

        #batch["attention_mask"] = batch["input_features"].ne(self.processor.tokenizer.pad_token_id).long()

        # Extrai e preenche os rótulos.
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, padding="longest", return_tensors="pt"
        )

        # Substitui os tokens de preenchimento por -100 para ignorar a perda.
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)

        # Remove o token de início do decoder, se já adicionado.
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch




In [12]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [13]:
import evaluate

metric = evaluate.load("wer")

In [14]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    #Substitua -100 pelo ID do token de preenchimento
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    #Nós não queremos agrupar tokens ao calcular as métricas.
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

In [17]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="../Fine-Tunning-medium-v1.2 ",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=3e-6,
    max_steps=250,
    gradient_checkpointing = True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=1,
    predict_with_generate=True,
    generation_max_length=50,
    save_steps=50,
    eval_steps=50,
    logging_steps=25 ,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    #push_to_hub = True,
    #deepspeed = "/home/bruno-dev/Documents/projeto-fala-texto/fine-tuning/ds_config.json"
)

In [18]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset = data['train'],
    eval_dataset = data['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

/home/fala-texto-server2/anaconda3/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [ ]:
model.save_pretrained("/mnt/hd/v1-medium/whisper_finetuned-medium-v1.2")
processor.save_pretrained("/mnt/hd/v1-medium/whisper_finetuned-medium-v1.2")

-- Realizar Avaliação do Treinamento

In [ ]:
results = trainer.evaluate()
print(f"Word Error Rate: {results['eval_wer']}")